# 05 — Silver Data Profiling

## Purpose

Bronze currently preserves each GH Archive event as raw JSON.

Before building the Silver Lakeflow pipeline, this notebook investigates the real
February 2024 data so that cleaning rules are based on evidence rather than assumptions.

We will check:

1. Event type distribution
2. JSON validity
3. Actual action and review-state values
4. General profiles of the 9 retained event types
5. Intended datatype compatibility
6. NULL coverage
7. Duplicate event IDs
8. Logical inconsistencies
9. Release-window edge cases
10. Final Silver table contracts

No Bronze data is modified in this notebook.

In [0]:
from pyspark.sql import functions as F
from datetime import timedelta


BRONZE_TABLE = "github_lakehouse.bronze.github_events_raw"


retained_event_types = [
    "PushEvent",
    "PullRequestEvent",
    "IssuesEvent",
    "IssueCommentEvent",
    "PullRequestReviewEvent",
    "PullRequestReviewCommentEvent",
    "WatchEvent",
    "ForkEvent",
    "ReleaseEvent"
]


dropped_event_types = [
    "CommitCommentEvent",
    "CreateEvent",
    "DeleteEvent",
    "GollumEvent",
    "MemberEvent",
    "PublicEvent"
]


bronze_df = spark.table(BRONZE_TABLE)

## 1. Event Type Distribution

First, we confirm which GitHub event types actually exist in Bronze and how many
records each event type contains.

The 9 retained event types will later become Silver tables.

The other 6 event types are valid data, but they are outside the scope of the
six Gold products. They remain safely preserved in Bronze.

In [0]:
json_base_df = (
    bronze_df
    .select(
        F.get_json_object("raw_json", "$").isNotNull().alias("is_valid_json"),
        F.get_json_object("raw_json", "$.type").alias("event_type")
    )
)

# Materialize the cached DataFrame once.
json_base_df.count()

38555222

In [0]:
event_counts = (
    json_base_df
    .groupBy("event_type")
    .count()
    .withColumn(
        "silver_decision",
        F.when(
            F.col("event_type").isin(retained_event_types),
            F.lit("KEEP")
        )
        .when(
            F.col("event_type").isin(dropped_event_types),
            F.lit("DROP")
        )
        .otherwise(F.lit("CHECK"))
    )
    .orderBy(F.desc("count"))
)

display(event_counts)

event_type,count,silver_decision
PushEvent,26537024,KEEP
CreateEvent,3778827,DROP
PullRequestEvent,2173964,KEEP
WatchEvent,1380565,KEEP
IssueCommentEvent,1357327,KEEP
DeleteEvent,874499,DROP
PullRequestReviewEvent,717285,KEEP
IssuesEvent,548959,KEEP
PullRequestReviewCommentEvent,422917,KEEP
ForkEvent,326419,KEEP


## 2. JSON Validity Check

Bronze intentionally stores the original event as a raw JSON string.

Before parsing Silver fields, we verify whether those strings contain valid JSON.

We separate two different problems:

- malformed JSON
- valid JSON that is missing the `type` field

Those are not the same issue.

In [0]:
json_summary = (
    json_base_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(F.col("is_valid_json"), 1).otherwise(0)
        ).alias("valid_json_rows"),

        F.sum(
            F.when(~F.col("is_valid_json"), 1).otherwise(0)
        ).alias("invalid_json_rows"),

        F.sum(
            F.when(
                F.col("is_valid_json") &
                F.col("event_type").isNull(),
                1
            ).otherwise(0)
        ).alias("valid_json_missing_event_type")
    )
)

display(json_summary)

total_rows,valid_json_rows,invalid_json_rows,valid_json_missing_event_type
38555222,38555222,0,0


In [0]:
invalid_json_count = (
    json_summary
    .select("invalid_json_rows")
    .first()["invalid_json_rows"]
)

if invalid_json_count > 0:
    invalid_json_sample = (
        bronze_df
        .filter(
            F.get_json_object("raw_json", "$").isNull()
        )
        .select(
            "raw_json",
            "_source_file",
            "_source_date"
        )
        .limit(20)
    )

    display(invalid_json_sample)

else:
    print("No malformed JSON records found.")

No malformed JSON records found.


## 3. Extract Fields Needed for Silver Profiling

The raw GitHub JSON contains hundreds of possible nested fields.

We do NOT flatten everything.

For profiling, we extract:

- common event fields
- the event-specific fields required by Gold
- Bronze lineage metadata

The extracted values remain mostly as strings at this stage so that we can separately
check whether conversion into Silver datatypes succeeds.

In [0]:
profile_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------
        # Common fields
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.id").alias("event_id_raw"),
        F.get_json_object("raw_json", "$.type").alias("event_type"),
        F.get_json_object("raw_json", "$.created_at").alias("event_timestamp_raw"),

        F.get_json_object("raw_json", "$.actor.id").alias("actor_id_raw"),
        F.get_json_object("raw_json", "$.actor.login").alias("actor_login_raw"),

        F.get_json_object("raw_json", "$.repo.id").alias("repo_id_raw"),
        F.get_json_object("raw_json", "$.repo.name").alias("repo_name_raw"),


        # ---------------------------------------------------------
        # Shared payload.action
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.action").alias("action_raw"),


        # ---------------------------------------------------------
        # PushEvent
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.size")
            .alias("push_commit_count_raw"),

        F.get_json_object("raw_json", "$.payload.distinct_size")
            .alias("push_distinct_commit_count_raw"),


        # ---------------------------------------------------------
        # PullRequestEvent
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.number")
            .alias("pr_number_event_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.id")
            .alias("pr_id_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.number")
            .alias("pr_number_nested_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.created_at")
            .alias("pr_created_at_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.closed_at")
            .alias("pr_closed_at_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.merged_at")
            .alias("pr_merged_at_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.merged")
            .alias("pr_merged_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.commits")
            .alias("pr_commits_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.additions")
            .alias("pr_additions_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.deletions")
            .alias("pr_deletions_raw"),

        F.get_json_object("raw_json", "$.payload.pull_request.changed_files")
            .alias("pr_changed_files_raw"),


        # ---------------------------------------------------------
        # IssuesEvent / IssueCommentEvent
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.issue.id")
            .alias("issue_id_raw"),

        F.get_json_object("raw_json", "$.payload.issue.number")
            .alias("issue_number_raw"),

        F.get_json_object("raw_json", "$.payload.issue.created_at")
            .alias("issue_created_at_raw"),

        F.get_json_object("raw_json", "$.payload.issue.closed_at")
            .alias("issue_closed_at_raw"),

        F.get_json_object("raw_json", "$.payload.issue.pull_request.url")
            .alias("issue_pr_url_raw"),


        # ---------------------------------------------------------
        # Comment Events
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.comment.id")
            .alias("comment_id_raw"),


        # ---------------------------------------------------------
        # PullRequestReviewEvent
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.review.id")
            .alias("review_id_raw"),

        F.get_json_object("raw_json", "$.payload.review.state")
            .alias("review_state_raw"),


        # ---------------------------------------------------------
        # ReleaseEvent
        # ---------------------------------------------------------

        F.get_json_object("raw_json", "$.payload.release.id")
            .alias("release_id_raw"),

        F.get_json_object("raw_json", "$.payload.release.tag_name")
            .alias("release_tag_raw"),

        F.get_json_object("raw_json", "$.payload.release.published_at")
            .alias("release_published_at_raw"),

        F.get_json_object("raw_json", "$.payload.release.draft")
            .alias("release_draft_raw"),

        F.get_json_object("raw_json", "$.payload.release.prerelease")
            .alias("release_prerelease_raw"),


        # ---------------------------------------------------------
        # Bronze lineage
        # ---------------------------------------------------------

        "_source_file",
        "_source_date",
        "_ingested_at"
    )

    # Only Silver-scoped event types are needed from this point onward.
    .filter(
        F.col("event_type").isin(retained_event_types)
    )
)


profile_row_count = profile_df.count()

print(f"Retained event rows available for profiling: {profile_row_count:,}")

Retained event rows available for profiling: 33,623,876


## 4. Discover Actual Action and Review-State Values

GitHub event documentation tells us what values may exist, but Silver rules should be
based on the actual February 2024 GH Archive data.

We therefore profile:

- `payload.action`
- `payload.review.state`

before hard-coding any assumptions.

In [0]:
action_event_types = [
    "PullRequestEvent",
    "IssuesEvent",
    "IssueCommentEvent",
    "PullRequestReviewEvent",
    "PullRequestReviewCommentEvent",
    "WatchEvent",
    "ReleaseEvent"
]


action_values = (
    profile_df
    .filter(
        F.col("event_type").isin(action_event_types)
    )
    .groupBy(
        "event_type",
        "action_raw"
    )
    .count()
    .orderBy(
        "event_type",
        F.desc("count")
    )
)

display(action_values)

event_type,action_raw,count
IssueCommentEvent,created,1357325
IssueCommentEvent,null,2
IssuesEvent,opened,322476
IssuesEvent,closed,217901
IssuesEvent,reopened,8582
PullRequestEvent,opened,1126808
PullRequestEvent,closed,1031732
PullRequestEvent,reopened,15424
PullRequestReviewCommentEvent,created,422917
PullRequestReviewEvent,created,717285


In [0]:
review_state_values = (
    profile_df
    .filter(
        F.col("event_type") == "PullRequestReviewEvent"
    )
    .groupBy("review_state_raw")
    .count()
    .orderBy(F.desc("count"))
)

display(review_state_values)

review_state_raw,count
approved,339097
commented,323312
changes_requested,43110
dismissed,11744
pending,22


### IssueCommentEvent Target Classification

`IssueCommentEvent` is slightly unusual.

A comment can belong to:

- a normal issue
- a pull-request conversation

If `payload.issue.pull_request.url` exists, the target is a pull request.
Otherwise, the target is an issue.

In [0]:
issue_comment_targets = (
    profile_df
    .filter(
        F.col("event_type") == "IssueCommentEvent"
    )
    .withColumn(
        "comment_target_type",

        F.when(
            F.col("issue_id_raw").isNull(),
            F.lit(None)
        )

        .when(
            F.col("issue_pr_url_raw").isNotNull(),
            F.lit("pull_request")
        )

        .otherwise(
            F.lit("issue")
        )
    )
    .groupBy("comment_target_type")
    .count()
    .orderBy(F.desc("count"))
)

display(issue_comment_targets)

comment_target_type,count
pull_request,838179
issue,519146
null,2


## 5. General Profile of the 9 Retained Event Types

Before deeper quality checks, we create a basic profile for each retained event type.

This shows:

- total records
- distinct event IDs
- distinct actors
- distinct repositories
- number of source files
- source-date coverage

In [0]:
event_overview = (
    profile_df
    .groupBy("event_type")
    .agg(

        F.count("*").alias("row_count"),

        F.countDistinct("event_id_raw")
            .alias("distinct_event_ids"),

        F.countDistinct("actor_id_raw")
            .alias("distinct_actors"),

        F.countDistinct("repo_id_raw")
            .alias("distinct_repositories"),

        F.countDistinct("_source_file")
            .alias("source_files"),

        F.min("_source_date")
            .alias("first_source_date"),

        F.max("_source_date")
            .alias("last_source_date")
    )
    .orderBy(F.desc("row_count"))
)

display(event_overview)

event_type,row_count,distinct_event_ids,distinct_actors,distinct_repositories,source_files,first_source_date,last_source_date
PushEvent,26537024,26536967,1361409,2153164,168,2024-02-01,2024-02-07
PullRequestEvent,2173964,2173958,293885,403265,168,2024-02-01,2024-02-07
WatchEvent,1380565,1380559,561691,464262,168,2024-02-01,2024-02-07
IssueCommentEvent,1357327,1357324,220790,191974,168,2024-02-01,2024-02-07
PullRequestReviewEvent,717285,717277,108692,83953,168,2024-02-01,2024-02-07
IssuesEvent,548959,548956,162926,117087,168,2024-02-01,2024-02-07
PullRequestReviewCommentEvent,422917,422917,62901,34248,168,2024-02-01,2024-02-07
ForkEvent,326419,326417,211859,162616,168,2024-02-01,2024-02-07
ReleaseEvent,159416,159416,34822,62125,168,2024-02-01,2024-02-07


## 6. Create a Temporary Typed View

Silver will use proper Spark datatypes instead of keeping everything as JSON strings.

We now attempt the exact conversions planned for Silver.

`try_cast()` is intentionally used here.

If a value cannot be converted, `try_cast()` returns NULL instead of stopping the notebook.
That allows us to count datatype failures safely.

In [0]:
typed_df = (
    profile_df

    # ============================================================
    # Common fields
    # ============================================================

    .withColumn(
        "event_id",
        F.col("event_id_raw")
    )

    .withColumn(
        "event_timestamp",
        F.expr("try_cast(event_timestamp_raw AS TIMESTAMP)")
    )

    .withColumn(
        "event_date",
        F.to_date("event_timestamp")
    )

    .withColumn(
        "actor_id",
        F.expr("try_cast(actor_id_raw AS BIGINT)")
    )

    .withColumn(
        "actor_login",
        F.col("actor_login_raw")
    )

    .withColumn(
        "repo_id",
        F.expr("try_cast(repo_id_raw AS BIGINT)")
    )

    .withColumn(
        "repo_name",
        F.col("repo_name_raw")
    )


    # ============================================================
    # PushEvent
    # ============================================================

    .withColumn(
        "push_commit_count",
        F.expr("try_cast(push_commit_count_raw AS BIGINT)")
    )

    .withColumn(
        "push_distinct_commit_count",
        F.expr("try_cast(push_distinct_commit_count_raw AS BIGINT)")
    )


    # ============================================================
    # PullRequestEvent
    # ============================================================

    .withColumn(
        "pr_action",
        F.when(
            F.col("event_type") == "PullRequestEvent",
            F.col("action_raw")
        )
    )

    .withColumn(
        "pr_number",
        F.when(
            F.col("event_type") == "PullRequestEvent",
            F.expr("try_cast(pr_number_event_raw AS BIGINT)")
        )
        .when(
            F.col("event_type").isin(
                "PullRequestReviewEvent",
                "PullRequestReviewCommentEvent"
            ),
            F.expr("try_cast(pr_number_nested_raw AS BIGINT)")
        )
    )

    .withColumn(
        "pr_id",
        F.expr("try_cast(pr_id_raw AS BIGINT)")
    )

    .withColumn(
        "pr_created_at",
        F.expr("try_cast(pr_created_at_raw AS TIMESTAMP)")
    )

    .withColumn(
        "pr_closed_at",
        F.expr("try_cast(pr_closed_at_raw AS TIMESTAMP)")
    )

    .withColumn(
        "pr_merged_at",
        F.expr("try_cast(pr_merged_at_raw AS TIMESTAMP)")
    )

    .withColumn(
        "pr_merged",
        F.expr("try_cast(pr_merged_raw AS BOOLEAN)")
    )

    .withColumn(
        "pr_commits",
        F.expr("try_cast(pr_commits_raw AS BIGINT)")
    )

    .withColumn(
        "pr_additions",
        F.expr("try_cast(pr_additions_raw AS BIGINT)")
    )

    .withColumn(
        "pr_deletions",
        F.expr("try_cast(pr_deletions_raw AS BIGINT)")
    )

    .withColumn(
        "pr_changed_files",
        F.expr("try_cast(pr_changed_files_raw AS BIGINT)")
    )


    # ============================================================
    # IssuesEvent
    # ============================================================

    .withColumn(
        "issue_action",
        F.when(
            F.col("event_type") == "IssuesEvent",
            F.col("action_raw")
        )
    )

    .withColumn(
        "issue_id",
        F.expr("try_cast(issue_id_raw AS BIGINT)")
    )

    .withColumn(
        "issue_number",
        F.expr("try_cast(issue_number_raw AS BIGINT)")
    )

    .withColumn(
        "issue_created_at",
        F.expr("try_cast(issue_created_at_raw AS TIMESTAMP)")
    )

    .withColumn(
        "issue_closed_at",
        F.expr("try_cast(issue_closed_at_raw AS TIMESTAMP)")
    )


    # ============================================================
    # Comment events
    # ============================================================

    .withColumn(
        "comment_action",
        F.when(
            F.col("event_type").isin(
                "IssueCommentEvent",
                "PullRequestReviewCommentEvent"
            ),
            F.col("action_raw")
        )
    )

    .withColumn(
        "comment_id",
        F.expr("try_cast(comment_id_raw AS BIGINT)")
    )

    .withColumn(
        "comment_target_type",

        F.when(
            F.col("event_type") != "IssueCommentEvent",
            F.lit(None)
        )

        .when(
            F.col("issue_id").isNull(),
            F.lit(None)
        )

        .when(
            F.col("issue_pr_url_raw").isNotNull(),
            F.lit("pull_request")
        )

        .otherwise(
            F.lit("issue")
        )
    )


    # ============================================================
    # PullRequestReviewEvent
    # ============================================================

    .withColumn(
        "review_action",
        F.when(
            F.col("event_type") == "PullRequestReviewEvent",
            F.col("action_raw")
        )
    )

    .withColumn(
        "review_id",
        F.expr("try_cast(review_id_raw AS BIGINT)")
    )

    .withColumn(
        "review_state",
        F.col("review_state_raw")
    )


    # ============================================================
    # WatchEvent
    # ============================================================

    .withColumn(
        "watch_action",
        F.when(
            F.col("event_type") == "WatchEvent",
            F.col("action_raw")
        )
    )


    # ============================================================
    # ReleaseEvent
    # ============================================================

    .withColumn(
        "release_action",
        F.when(
            F.col("event_type") == "ReleaseEvent",
            F.col("action_raw")
        )
    )

    .withColumn(
        "release_id",
        F.expr("try_cast(release_id_raw AS BIGINT)")
    )

    .withColumn(
        "release_tag",
        F.col("release_tag_raw")
    )

    .withColumn(
        "release_published_at",
        F.expr("try_cast(release_published_at_raw AS TIMESTAMP)")
    )

    .withColumn(
        "release_draft",
        F.expr("try_cast(release_draft_raw AS BOOLEAN)")
    )

    .withColumn(
        "release_prerelease",
        F.expr("try_cast(release_prerelease_raw AS BOOLEAN)")
    )
)


typed_row_count = typed_df.count()

print(f"Typed profiling rows: {typed_row_count:,}")

Typed profiling rows: 33,623,876


## Silver Contract Definitions

The following definitions come from the locked Silver schema reference.

They are not yet creating physical Silver tables.

They simply describe:

- final Silver column name
- exact JSON source
- intended Spark datatype

In [0]:
common_contract_fields = [
    {
        "field": "event_id",
        "source": "id",
        "type": "STRING"
    },
    {
        "field": "event_type",
        "source": "type",
        "type": "STRING"
    },
    {
        "field": "event_timestamp",
        "source": "created_at",
        "type": "TIMESTAMP"
    },
    {
        "field": "event_date",
        "source": "derived: DATE(event_timestamp)",
        "type": "DATE"
    },
    {
        "field": "actor_id",
        "source": "actor.id",
        "type": "BIGINT"
    },
    {
        "field": "actor_login",
        "source": "actor.login",
        "type": "STRING"
    },
    {
        "field": "repo_id",
        "source": "repo.id",
        "type": "BIGINT"
    },
    {
        "field": "repo_name",
        "source": "repo.name",
        "type": "STRING"
    }
]


lineage_contract_fields = [
    {
        "field": "_source_file",
        "source": "Bronze column",
        "type": "STRING"
    },
    {
        "field": "_source_date",
        "source": "Bronze column",
        "type": "DATE"
    },
    {
        "field": "_ingested_at",
        "source": "Bronze column",
        "type": "TIMESTAMP"
    }
]

In [0]:
event_specific_contract_fields = {

    "PushEvent": [
        {
            "field": "push_commit_count",
            "source": "payload.size",
            "type": "BIGINT"
        },
        {
            "field": "push_distinct_commit_count",
            "source": "payload.distinct_size",
            "type": "BIGINT"
        }
    ],


    "PullRequestEvent": [
        {
            "field": "pr_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "pr_number",
            "source": "payload.number",
            "type": "BIGINT"
        },
        {
            "field": "pr_id",
            "source": "payload.pull_request.id",
            "type": "BIGINT"
        },
        {
            "field": "pr_created_at",
            "source": "payload.pull_request.created_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_closed_at",
            "source": "payload.pull_request.closed_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_merged_at",
            "source": "payload.pull_request.merged_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_merged",
            "source": "payload.pull_request.merged",
            "type": "BOOLEAN"
        },
        {
            "field": "pr_commits",
            "source": "payload.pull_request.commits",
            "type": "BIGINT"
        },
        {
            "field": "pr_additions",
            "source": "payload.pull_request.additions",
            "type": "BIGINT"
        },
        {
            "field": "pr_deletions",
            "source": "payload.pull_request.deletions",
            "type": "BIGINT"
        },
        {
            "field": "pr_changed_files",
            "source": "payload.pull_request.changed_files",
            "type": "BIGINT"
        }
    ],


    "IssuesEvent": [
        {
            "field": "issue_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "issue_id",
            "source": "payload.issue.id",
            "type": "BIGINT"
        },
        {
            "field": "issue_number",
            "source": "payload.issue.number",
            "type": "BIGINT"
        },
        {
            "field": "issue_created_at",
            "source": "payload.issue.created_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "issue_closed_at",
            "source": "payload.issue.closed_at",
            "type": "TIMESTAMP"
        }
    ],


    "IssueCommentEvent": [
        {
            "field": "comment_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "issue_id",
            "source": "payload.issue.id",
            "type": "BIGINT"
        },
        {
            "field": "issue_number",
            "source": "payload.issue.number",
            "type": "BIGINT"
        },
        {
            "field": "comment_target_type",
            "source": "derived from payload.issue.pull_request.url",
            "type": "STRING"
        },
        {
            "field": "comment_id",
            "source": "payload.comment.id",
            "type": "BIGINT"
        }
    ],


    "PullRequestReviewEvent": [
        {
            "field": "review_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "pr_id",
            "source": "payload.pull_request.id",
            "type": "BIGINT"
        },
        {
            "field": "pr_number",
            "source": "payload.pull_request.number",
            "type": "BIGINT"
        },
        {
            "field": "review_id",
            "source": "payload.review.id",
            "type": "BIGINT"
        },
        {
            "field": "review_state",
            "source": "payload.review.state",
            "type": "STRING"
        }
    ],


    "PullRequestReviewCommentEvent": [
        {
            "field": "comment_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "pr_id",
            "source": "payload.pull_request.id",
            "type": "BIGINT"
        },
        {
            "field": "pr_number",
            "source": "payload.pull_request.number",
            "type": "BIGINT"
        },
        {
            "field": "comment_id",
            "source": "payload.comment.id",
            "type": "BIGINT"
        }
    ],


    "WatchEvent": [
        {
            "field": "watch_action",
            "source": "payload.action",
            "type": "STRING"
        }
    ],


    "ForkEvent": [
        # No event-specific fields are required.
    ],


    "ReleaseEvent": [
        {
            "field": "release_action",
            "source": "payload.action",
            "type": "STRING"
        },
        {
            "field": "release_id",
            "source": "payload.release.id",
            "type": "BIGINT"
        },
        {
            "field": "release_tag",
            "source": "payload.release.tag_name",
            "type": "STRING"
        },
        {
            "field": "release_published_at",
            "source": "payload.release.published_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "release_draft",
            "source": "payload.release.draft",
            "type": "BOOLEAN"
        },
        {
            "field": "release_prerelease",
            "source": "payload.release.prerelease",
            "type": "BOOLEAN"
        }
    ]
}

In [0]:
silver_table_names = {
    "PushEvent": "github_lakehouse.silver.push_events",
    "PullRequestEvent": "github_lakehouse.silver.pull_request_events",
    "IssuesEvent": "github_lakehouse.silver.issue_events",
    "IssueCommentEvent": "github_lakehouse.silver.issue_comment_events",
    "PullRequestReviewEvent": "github_lakehouse.silver.pull_request_review_events",
    "PullRequestReviewCommentEvent": "github_lakehouse.silver.pull_request_review_comment_events",
    "WatchEvent": "github_lakehouse.silver.watch_events",
    "ForkEvent": "github_lakehouse.silver.fork_events",
    "ReleaseEvent": "github_lakehouse.silver.release_events"
}


gold_dependencies = {
    "PushEvent": "G3, G5, G6",
    "PullRequestEvent": "G1, G3, G5, G6",
    "IssuesEvent": "G2, G3, G5, G6",
    "IssueCommentEvent": "G3, G4, G6",
    "PullRequestReviewEvent": "G3, G4, G6",
    "PullRequestReviewCommentEvent": "G3, G4, G6",
    "WatchEvent": "G5, G6",
    "ForkEvent": "G5, G6",
    "ReleaseEvent": "G5, G6"
}

## 7. Intended Datatype Validation

Before Silver is built, we check whether the source values can actually be converted
into the datatypes defined in the Silver schema.

Example:

`actor.id`
STRING in raw JSON
→ BIGINT in Silver

A datatype failure means:

- the source value exists
- but `try_cast()` could not convert it

This is different from a legitimate source NULL.

In [0]:
common_cast_checks = [
    {
        "field": "event_timestamp",
        "raw": "event_timestamp_raw",
        "typed": "event_timestamp",
        "type": "TIMESTAMP"
    },
    {
        "field": "actor_id",
        "raw": "actor_id_raw",
        "typed": "actor_id",
        "type": "BIGINT"
    },
    {
        "field": "repo_id",
        "raw": "repo_id_raw",
        "typed": "repo_id",
        "type": "BIGINT"
    }
]


event_specific_cast_checks = {

    "PushEvent": [
        {
            "field": "push_commit_count",
            "raw": "push_commit_count_raw",
            "typed": "push_commit_count",
            "type": "BIGINT"
        },
        {
            "field": "push_distinct_commit_count",
            "raw": "push_distinct_commit_count_raw",
            "typed": "push_distinct_commit_count",
            "type": "BIGINT"
        }
    ],


    "PullRequestEvent": [
        {
            "field": "pr_number",
            "raw": "pr_number_event_raw",
            "typed": "pr_number",
            "type": "BIGINT"
        },
        {
            "field": "pr_id",
            "raw": "pr_id_raw",
            "typed": "pr_id",
            "type": "BIGINT"
        },
        {
            "field": "pr_created_at",
            "raw": "pr_created_at_raw",
            "typed": "pr_created_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_closed_at",
            "raw": "pr_closed_at_raw",
            "typed": "pr_closed_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_merged_at",
            "raw": "pr_merged_at_raw",
            "typed": "pr_merged_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "pr_merged",
            "raw": "pr_merged_raw",
            "typed": "pr_merged",
            "type": "BOOLEAN"
        },
        {
            "field": "pr_commits",
            "raw": "pr_commits_raw",
            "typed": "pr_commits",
            "type": "BIGINT"
        },
        {
            "field": "pr_additions",
            "raw": "pr_additions_raw",
            "typed": "pr_additions",
            "type": "BIGINT"
        },
        {
            "field": "pr_deletions",
            "raw": "pr_deletions_raw",
            "typed": "pr_deletions",
            "type": "BIGINT"
        },
        {
            "field": "pr_changed_files",
            "raw": "pr_changed_files_raw",
            "typed": "pr_changed_files",
            "type": "BIGINT"
        }
    ],


    "IssuesEvent": [
        {
            "field": "issue_id",
            "raw": "issue_id_raw",
            "typed": "issue_id",
            "type": "BIGINT"
        },
        {
            "field": "issue_number",
            "raw": "issue_number_raw",
            "typed": "issue_number",
            "type": "BIGINT"
        },
        {
            "field": "issue_created_at",
            "raw": "issue_created_at_raw",
            "typed": "issue_created_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "issue_closed_at",
            "raw": "issue_closed_at_raw",
            "typed": "issue_closed_at",
            "type": "TIMESTAMP"
        }
    ],


    "IssueCommentEvent": [
        {
            "field": "issue_id",
            "raw": "issue_id_raw",
            "typed": "issue_id",
            "type": "BIGINT"
        },
        {
            "field": "issue_number",
            "raw": "issue_number_raw",
            "typed": "issue_number",
            "type": "BIGINT"
        },
        {
            "field": "comment_id",
            "raw": "comment_id_raw",
            "typed": "comment_id",
            "type": "BIGINT"
        }
    ],


    "PullRequestReviewEvent": [
        {
            "field": "pr_id",
            "raw": "pr_id_raw",
            "typed": "pr_id",
            "type": "BIGINT"
        },
        {
            "field": "pr_number",
            "raw": "pr_number_nested_raw",
            "typed": "pr_number",
            "type": "BIGINT"
        },
        {
            "field": "review_id",
            "raw": "review_id_raw",
            "typed": "review_id",
            "type": "BIGINT"
        }
    ],


    "PullRequestReviewCommentEvent": [
        {
            "field": "pr_id",
            "raw": "pr_id_raw",
            "typed": "pr_id",
            "type": "BIGINT"
        },
        {
            "field": "pr_number",
            "raw": "pr_number_nested_raw",
            "typed": "pr_number",
            "type": "BIGINT"
        },
        {
            "field": "comment_id",
            "raw": "comment_id_raw",
            "typed": "comment_id",
            "type": "BIGINT"
        }
    ],


    "WatchEvent": [],

    "ForkEvent": [],


    "ReleaseEvent": [
        {
            "field": "release_id",
            "raw": "release_id_raw",
            "typed": "release_id",
            "type": "BIGINT"
        },
        {
            "field": "release_published_at",
            "raw": "release_published_at_raw",
            "typed": "release_published_at",
            "type": "TIMESTAMP"
        },
        {
            "field": "release_draft",
            "raw": "release_draft_raw",
            "typed": "release_draft",
            "type": "BOOLEAN"
        },
        {
            "field": "release_prerelease",
            "raw": "release_prerelease_raw",
            "typed": "release_prerelease",
            "type": "BOOLEAN"
        }
    ]
}

In [0]:
cast_profile_rows = []


for event_type in retained_event_types:

    checks = (
        common_cast_checks
        + event_specific_cast_checks[event_type]
    )

    event_df = typed_df.filter(
        F.col("event_type") == event_type
    )

    expressions = []

    for check in checks:

        field = check["field"]
        raw_col = check["raw"]
        typed_col = check["typed"]

        expressions.append(
            F.sum(
                F.when(
                    F.col(raw_col).isNotNull(),
                    1
                ).otherwise(0)
            ).alias(f"{field}__source_non_null")
        )

        expressions.append(
            F.sum(
                F.when(
                    F.col(raw_col).isNotNull()
                    & F.col(typed_col).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{field}__cast_failures")
        )


    result = (
        event_df
        .agg(*expressions)
        .first()
        .asDict()
    )


    for check in checks:

        field = check["field"]

        non_null = result[f"{field}__source_non_null"]
        failures = result[f"{field}__cast_failures"]

        if non_null == 0:
            failure_pct = 0.0
        else:
            failure_pct = round(
                failures / non_null * 100,
                4
            )


        cast_profile_rows.append(
            (
                event_type,
                field,
                check["type"],
                non_null,
                failures,
                failure_pct
            )
        )

In [0]:
cast_profile_df = spark.createDataFrame(
    cast_profile_rows,
    [
        "event_type",
        "field",
        "target_type",
        "source_non_null",
        "cast_failures",
        "cast_failure_pct"
    ]
)


display(
    cast_profile_df
    .orderBy(
        "event_type",
        "field"
    )
)

event_type,field,target_type,source_non_null,cast_failures,cast_failure_pct
ForkEvent,actor_id,BIGINT,326419,0,0.0
ForkEvent,event_timestamp,TIMESTAMP,326419,0,0.0
ForkEvent,repo_id,BIGINT,326419,0,0.0
IssueCommentEvent,actor_id,BIGINT,1357327,0,0.0
IssueCommentEvent,comment_id,BIGINT,1357325,0,0.0
IssueCommentEvent,event_timestamp,TIMESTAMP,1357327,0,0.0
IssueCommentEvent,issue_id,BIGINT,1357325,0,0.0
IssueCommentEvent,issue_number,BIGINT,1357325,0,0.0
IssueCommentEvent,repo_id,BIGINT,1357327,0,0.0
IssuesEvent,actor_id,BIGINT,548959,0,0.0


## 8. NULL Coverage

NULL does not automatically mean bad data.

Examples:

- an open pull request can legitimately have `pr_closed_at = NULL`
- an unmerged pull request can legitimately have `pr_merged_at = NULL`
- an open issue can legitimately have `issue_closed_at = NULL`

This profile tells us how frequently each planned Silver column is NULL.

The results will later help us classify fields as:

- required
- legitimately nullable
- conditionally required

In [0]:
null_profile_rows = []


for event_type in retained_event_types:

    fields = (
        common_contract_fields
        + lineage_contract_fields
        + event_specific_contract_fields[event_type]
    )


    event_df = typed_df.filter(
        F.col("event_type") == event_type
    )


    expressions = [
        F.count("*").alias("_total_rows")
    ]


    for field_info in fields:

        field = field_info["field"]

        expressions.append(
            F.sum(
                F.when(
                    F.col(field).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"null__{field}")
        )


        # Empty strings are also worth finding.
        if field_info["type"] == "STRING":

            expressions.append(
                F.sum(
                    F.when(
                        F.col(field).isNotNull()
                        & (F.trim(F.col(field)) == ""),
                        1
                    ).otherwise(0)
                ).alias(f"blank__{field}")
            )


    result = (
        event_df
        .agg(*expressions)
        .first()
        .asDict()
    )


    total_rows = result["_total_rows"]


    for field_info in fields:

        field = field_info["field"]

        null_count = result[f"null__{field}"]

        if field_info["type"] == "STRING":
            blank_count = result[f"blank__{field}"]
        else:
            blank_count = 0


        if total_rows == 0:
            null_pct = 0.0
        else:
            null_pct = round(
                null_count / total_rows * 100,
                4
            )


        null_profile_rows.append(
            (
                event_type,
                field,
                field_info["type"],
                total_rows,
                null_count,
                null_pct,
                blank_count
            )
        )

In [0]:
null_profile_df = spark.createDataFrame(
    null_profile_rows,
    [
        "event_type",
        "field",
        "silver_type",
        "total_rows",
        "null_count",
        "null_pct",
        "blank_string_count"
    ]
)


display(
    null_profile_df
    .orderBy(
        "event_type",
        "field"
    )
)

event_type,field,silver_type,total_rows,null_count,null_pct,blank_string_count
ForkEvent,_ingested_at,TIMESTAMP,326419,0,0.0,0
ForkEvent,_source_date,DATE,326419,0,0.0,0
ForkEvent,_source_file,STRING,326419,0,0.0,0
ForkEvent,actor_id,BIGINT,326419,0,0.0,0
ForkEvent,actor_login,STRING,326419,0,0.0,0
ForkEvent,event_date,DATE,326419,0,0.0,0
ForkEvent,event_id,STRING,326419,0,0.0,0
ForkEvent,event_timestamp,TIMESTAMP,326419,0,0.0,0
ForkEvent,event_type,STRING,326419,0,0.0,0
ForkEvent,repo_id,BIGINT,326419,0,0.0,0


In [0]:
display(
    null_profile_df
    .filter(
        (F.col("null_count") > 0)
        | (F.col("blank_string_count") > 0)
    )
    .orderBy(
        F.desc("null_pct")
    )
)

event_type,field,silver_type,total_rows,null_count,null_pct,blank_string_count
PullRequestEvent,pr_merged_at,TIMESTAMP,2173964,1369201,62.9818,0
IssuesEvent,issue_closed_at,TIMESTAMP,548959,330119,60.1355,0
PullRequestEvent,pr_closed_at,TIMESTAMP,2173964,1136804,52.2918,0
ReleaseEvent,release_published_at,TIMESTAMP,159416,278,0.1744,0
IssueCommentEvent,comment_action,STRING,1357327,2,1.0E-4,0
IssueCommentEvent,issue_id,BIGINT,1357327,2,1.0E-4,0
IssueCommentEvent,issue_number,BIGINT,1357327,2,1.0E-4,0
IssueCommentEvent,comment_target_type,STRING,1357327,2,1.0E-4,0
IssueCommentEvent,comment_id,BIGINT,1357327,2,1.0E-4,0
ReleaseEvent,release_tag,STRING,159416,0,0.0,1


## 9. Duplicate Event ID Profiling

The Silver layer should eventually have one trusted record per GitHub event ID.

Before choosing a deduplication strategy, we first measure whether duplicates exist.

We check:

- total records
- distinct event IDs
- NULL event IDs
- duplicated event IDs
- extra duplicate rows

In [0]:
duplicate_ids_df = (
    typed_df
    .filter(
        F.col("event_id").isNotNull()
    )
    .groupBy("event_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("event_type").alias("event_type_count"),
        F.collect_set("event_type").alias("event_types")
    )
    .filter(
        F.col("row_count") > 1
    )
)

In [0]:
event_duplicate_base = (
    typed_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("total_rows"),

        F.countDistinct("event_id")
            .alias("distinct_event_ids"),

        F.sum(
            F.when(
                F.col("event_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_event_ids")
    )
)

In [0]:
duplicate_event_rows = (
    typed_df
    .join(
        duplicate_ids_df.select("event_id"),
        on="event_id",
        how="inner"
    )
    .groupBy("event_type")
    .agg(
        F.countDistinct("event_id")
            .alias("duplicate_event_ids"),

        (
            F.count("*")
            - F.countDistinct("event_id")
        ).alias("extra_duplicate_rows")
    )
)

In [0]:
duplicate_summary = (
    event_duplicate_base
    .join(
        duplicate_event_rows,
        on="event_type",
        how="left"
    )
    .fillna(
        {
            "duplicate_event_ids": 0,
            "extra_duplicate_rows": 0
        }
    )
    .orderBy("event_type")
)


display(duplicate_summary)

event_type,total_rows,distinct_event_ids,null_event_ids,duplicate_event_ids,extra_duplicate_rows
ForkEvent,326419,326417,0,2,2
IssueCommentEvent,1357327,1357324,0,3,3
IssuesEvent,548959,548956,0,2,3
PullRequestEvent,2173964,2173958,0,5,6
PullRequestReviewCommentEvent,422917,422917,0,0,0
PullRequestReviewEvent,717285,717277,0,4,8
PushEvent,26537024,26536967,0,56,57
ReleaseEvent,159416,159416,0,0,0
WatchEvent,1380565,1380559,0,6,6


In [0]:
display(
    duplicate_ids_df
    .orderBy(
        F.desc("row_count")
    )
    .limit(20)
)

event_id,row_count,event_type_count,event_types
35398325768,6,1,List(PullRequestReviewEvent)
35420010273,3,1,List(PullRequestEvent)
35420010565,3,1,List(PushEvent)
35420010602,3,1,List(IssuesEvent)
35392445033,2,1,List(PushEvent)
35298014068,2,1,List(PushEvent)
35392445225,2,1,List(PushEvent)
35392467568,2,1,List(PushEvent)
35392445174,2,1,List(PushEvent)
35392459443,2,1,List(PushEvent)


In [0]:
sample_duplicate_ids = [
    row["event_id"]

    for row in (
        duplicate_ids_df
        .orderBy(F.desc("row_count"))
        .limit(20)
        .collect()
    )
]


if sample_duplicate_ids:

    duplicate_content_check = (
        bronze_df
        .select(
            F.get_json_object(
                "raw_json",
                "$.id"
            ).alias("event_id"),

            "raw_json"
        )
        .filter(
            F.col("event_id").isin(sample_duplicate_ids)
        )
        .groupBy("event_id")
        .agg(
            F.count("*").alias("rows"),

            F.countDistinct("raw_json")
                .alias("distinct_raw_json_versions")
        )
        .orderBy(
            F.desc("rows")
        )
    )

    display(duplicate_content_check)

else:
    print("No duplicate event IDs found.")

event_id,rows,distinct_raw_json_versions
35398325768,6,1
35420010565,3,1
35420010273,3,1
35420010602,3,1
35392458975,2,1
35392459030,2,1
35392459132,2,1
35392442537,2,1
35392459443,2,1
35392445203,2,1


## 10. Logical Consistency Checks

A record can:

- contain valid JSON
- have the correct datatype
- contain no unexpected NULLs

and still be logically suspicious.

Examples:

- PR merge timestamp before PR creation
- negative commit count
- issue closed before it was created

These checks help us identify those cases before writing Silver quality rules.

In [0]:
logical_checks = [

    # ------------------------------------------------------------
    # PushEvent
    # ------------------------------------------------------------

    (
        "PushEvent",
        "negative_push_commit_count",
        F.col("push_commit_count") < 0
    ),

    (
        "PushEvent",
        "negative_push_distinct_commit_count",
        F.col("push_distinct_commit_count") < 0
    ),

    (
        "PushEvent",
        "distinct_commits_greater_than_total_commits",
        F.col("push_distinct_commit_count") > F.col("push_commit_count")
    ),


    # ------------------------------------------------------------
    # PullRequestEvent
    # ------------------------------------------------------------

    (
        "PullRequestEvent",
        "pr_closed_before_created",
        F.col("pr_closed_at") < F.col("pr_created_at")
    ),

    (
        "PullRequestEvent",
        "pr_merged_before_created",
        F.col("pr_merged_at") < F.col("pr_created_at")
    ),

    (
        "PullRequestEvent",
        "merged_true_but_merged_at_null",
        (F.col("pr_merged") == True)
        & F.col("pr_merged_at").isNull()
    ),

    (
        "PullRequestEvent",
        "merged_false_but_merged_at_exists",
        (F.col("pr_merged") == False)
        & F.col("pr_merged_at").isNotNull()
    ),

    (
        "PullRequestEvent",
        "negative_pr_commits",
        F.col("pr_commits") < 0
    ),

    (
        "PullRequestEvent",
        "negative_pr_additions",
        F.col("pr_additions") < 0
    ),

    (
        "PullRequestEvent",
        "negative_pr_deletions",
        F.col("pr_deletions") < 0
    ),

    (
        "PullRequestEvent",
        "negative_pr_changed_files",
        F.col("pr_changed_files") < 0
    ),


    # ------------------------------------------------------------
    # IssuesEvent
    # ------------------------------------------------------------

    (
        "IssuesEvent",
        "issue_closed_before_created",
        F.col("issue_closed_at") < F.col("issue_created_at")
    ),


    # ------------------------------------------------------------
    # ReleaseEvent
    # ------------------------------------------------------------

    (
        "ReleaseEvent",
        "published_release_missing_published_at",
        (F.col("release_action") == "published")
        & F.col("release_published_at").isNull()
    )
]

In [0]:
logical_check_rows = []


for event_type, check_name, condition in logical_checks:

    problem_count = (
        typed_df
        .filter(
            (F.col("event_type") == event_type)
            & condition
        )
        .count()
    )


    logical_check_rows.append(
        (
            event_type,
            check_name,
            problem_count
        )
    )


logical_check_df = spark.createDataFrame(
    logical_check_rows,
    [
        "event_type",
        "check_name",
        "problem_count"
    ]
)


display(
    logical_check_df
    .orderBy(
        F.desc("problem_count")
    )
)

event_type,check_name,problem_count
PushEvent,distinct_commits_greater_than_total_commits,611
ReleaseEvent,published_release_missing_published_at,278
PushEvent,negative_push_commit_count,0
PushEvent,negative_push_distinct_commit_count,0
PullRequestEvent,pr_closed_before_created,0
PullRequestEvent,pr_merged_before_created,0
PullRequestEvent,merged_true_but_merged_at_null,0
PullRequestEvent,merged_false_but_merged_at_exists,0
PullRequestEvent,negative_pr_commits,0
PullRequestEvent,negative_pr_additions,0


### Source-Date Consistency

GH Archive source files are organized by date.

We compare the event timestamp's date against the Bronze `_source_date`.

A mismatch is not automatically deleted, but unexpected values should be investigated.

In [0]:
source_date_mismatch = (
    typed_df
    .filter(
        F.col("event_timestamp").isNotNull()
        & F.col("_source_date").isNotNull()
        & (F.col("event_date") != F.col("_source_date"))
    )
    .groupBy("event_type")
    .count()
    .orderBy(F.desc("count"))
)

display(source_date_mismatch)

event_type,count


## 11. Release Observation-Window Profiling

The Gold `release_activity_shift` table compares activity:

24 hours before a release
vs.
24 hours after a release

Our dataset contains only seven days.

Therefore, releases close to the start or end of the dataset do not have a complete
observation window.

We profile those releases now so Gold can later exclude them safely.

In [0]:
dataset_bounds = (
    typed_df
    .agg(
        F.min("event_timestamp")
            .alias("dataset_start"),

        F.max("event_timestamp")
            .alias("dataset_end")
    )
    .first()
)


dataset_start = dataset_bounds["dataset_start"]
dataset_end = dataset_bounds["dataset_end"]


print("Dataset start:", dataset_start)
print("Dataset end:  ", dataset_end)

Dataset start: 2024-02-01 00:00:00
Dataset end:   2024-02-07 23:59:59


In [0]:
safe_release_start = dataset_start + timedelta(hours=24)
safe_release_end = dataset_end - timedelta(hours=24)


print("Earliest release with full ±24h window:", safe_release_start)
print("Latest release with full ±24h window:  ", safe_release_end)

Earliest release with full ±24h window: 2024-02-02 00:00:00
Latest release with full ±24h window:   2024-02-06 23:59:59


In [0]:
release_window_profile = (
    typed_df
    .filter(
        (F.col("event_type") == "ReleaseEvent")
        & (F.col("release_action") == "published")
    )
    .withColumn(
        "has_complete_24h_window",

        (F.col("release_published_at") >= F.lit(safe_release_start))
        &
        (F.col("release_published_at") <= F.lit(safe_release_end))
    )
    .groupBy("has_complete_24h_window")
    .count()
)

display(release_window_profile)

has_complete_24h_window,count
null,278
true,108876
false,50262


## 12. Final Silver Contract Review

We now combine:

- the locked field mapping
- intended datatypes
- observed NULL rates
- blank-string counts
- datatype-cast failures
- Gold dependencies

This gives us one reviewable contract for each of the 9 planned Silver tables.

Important:

The notebook should NOT automatically decide that a field is invalid merely because
it contains NULLs.

NULL policy will be finalized after reviewing these results.

In [0]:
contract_rows = []


for event_type in retained_event_types:

    all_fields = (
        common_contract_fields
        + lineage_contract_fields
        + event_specific_contract_fields[event_type]
    )


    for position, field_info in enumerate(all_fields, start=1):

        contract_rows.append(
            (
                event_type,
                silver_table_names[event_type],
                position,
                field_info["field"],
                field_info["source"],
                field_info["type"],
                gold_dependencies[event_type]
            )
        )

In [0]:
contract_df = spark.createDataFrame(
    contract_rows,
    [
        "event_type",
        "silver_table",
        "field_order",
        "silver_field",
        "source_path",
        "silver_type",
        "gold_dependency"
    ]
)

In [0]:
contract_review_df = (
    contract_df

    .join(
        null_profile_df
        .select(
            "event_type",
            F.col("field").alias("silver_field"),
            "null_count",
            "null_pct",
            "blank_string_count"
        ),

        on=[
            "event_type",
            "silver_field"
        ],

        how="left"
    )

    .join(
        cast_profile_df
        .select(
            "event_type",
            F.col("field").alias("silver_field"),
            "cast_failures",
            "cast_failure_pct"
        ),

        on=[
            "event_type",
            "silver_field"
        ],

        how="left"
    )

    .fillna(
        {
            "cast_failures": 0,
            "cast_failure_pct": 0.0
        }
    )
)

In [0]:
display(
    contract_review_df
    .orderBy(
        "event_type",
        "field_order"
    )
)

event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
ForkEvent,event_id,github_lakehouse.silver.fork_events,1,id,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_type,github_lakehouse.silver.fork_events,2,type,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_timestamp,github_lakehouse.silver.fork_events,3,created_at,TIMESTAMP,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_date,github_lakehouse.silver.fork_events,4,derived: DATE(event_timestamp),DATE,"G5, G6",0,0.0,0,0,0.0
ForkEvent,actor_id,github_lakehouse.silver.fork_events,5,actor.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ForkEvent,actor_login,github_lakehouse.silver.fork_events,6,actor.login,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,repo_id,github_lakehouse.silver.fork_events,7,repo.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ForkEvent,repo_name,github_lakehouse.silver.fork_events,8,repo.name,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,_source_file,github_lakehouse.silver.fork_events,9,Bronze column,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,_source_date,github_lakehouse.silver.fork_events,10,Bronze column,DATE,"G5, G6",0,0.0,0,0,0.0


In [0]:
for event_type in retained_event_types:

    print("=" * 90)
    print(event_type)
    print(silver_table_names[event_type])
    print("=" * 90)

    display(
        contract_review_df
        .filter(
            F.col("event_type") == event_type
        )
        .orderBy("field_order")
    )

PushEvent
github_lakehouse.silver.push_events


event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
PushEvent,event_id,github_lakehouse.silver.push_events,1,id,STRING,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,event_type,github_lakehouse.silver.push_events,2,type,STRING,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,event_timestamp,github_lakehouse.silver.push_events,3,created_at,TIMESTAMP,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,event_date,github_lakehouse.silver.push_events,4,derived: DATE(event_timestamp),DATE,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,actor_id,github_lakehouse.silver.push_events,5,actor.id,BIGINT,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,actor_login,github_lakehouse.silver.push_events,6,actor.login,STRING,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,repo_id,github_lakehouse.silver.push_events,7,repo.id,BIGINT,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,repo_name,github_lakehouse.silver.push_events,8,repo.name,STRING,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,_source_file,github_lakehouse.silver.push_events,9,Bronze column,STRING,"G3, G5, G6",0,0.0,0,0,0.0
PushEvent,_source_date,github_lakehouse.silver.push_events,10,Bronze column,DATE,"G3, G5, G6",0,0.0,0,0,0.0


PullRequestEvent
github_lakehouse.silver.pull_request_events


event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
PullRequestEvent,event_id,github_lakehouse.silver.pull_request_events,1,id,STRING,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,event_type,github_lakehouse.silver.pull_request_events,2,type,STRING,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,event_timestamp,github_lakehouse.silver.pull_request_events,3,created_at,TIMESTAMP,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,event_date,github_lakehouse.silver.pull_request_events,4,derived: DATE(event_timestamp),DATE,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,actor_id,github_lakehouse.silver.pull_request_events,5,actor.id,BIGINT,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,actor_login,github_lakehouse.silver.pull_request_events,6,actor.login,STRING,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,repo_id,github_lakehouse.silver.pull_request_events,7,repo.id,BIGINT,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,repo_name,github_lakehouse.silver.pull_request_events,8,repo.name,STRING,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,_source_file,github_lakehouse.silver.pull_request_events,9,Bronze column,STRING,"G1, G3, G5, G6",0,0.0,0,0,0.0
PullRequestEvent,_source_date,github_lakehouse.silver.pull_request_events,10,Bronze column,DATE,"G1, G3, G5, G6",0,0.0,0,0,0.0


IssuesEvent
github_lakehouse.silver.issue_events


...

event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
WatchEvent,event_id,github_lakehouse.silver.watch_events,1,id,STRING,"G5, G6",0,0.0,0,0,0.0
WatchEvent,event_type,github_lakehouse.silver.watch_events,2,type,STRING,"G5, G6",0,0.0,0,0,0.0
WatchEvent,event_timestamp,github_lakehouse.silver.watch_events,3,created_at,TIMESTAMP,"G5, G6",0,0.0,0,0,0.0
WatchEvent,event_date,github_lakehouse.silver.watch_events,4,derived: DATE(event_timestamp),DATE,"G5, G6",0,0.0,0,0,0.0
WatchEvent,actor_id,github_lakehouse.silver.watch_events,5,actor.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
WatchEvent,actor_login,github_lakehouse.silver.watch_events,6,actor.login,STRING,"G5, G6",0,0.0,0,0,0.0
WatchEvent,repo_id,github_lakehouse.silver.watch_events,7,repo.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
WatchEvent,repo_name,github_lakehouse.silver.watch_events,8,repo.name,STRING,"G5, G6",0,0.0,0,0,0.0
WatchEvent,_source_file,github_lakehouse.silver.watch_events,9,Bronze column,STRING,"G5, G6",0,0.0,0,0,0.0
WatchEvent,_source_date,github_lakehouse.silver.watch_events,10,Bronze column,DATE,"G5, G6",0,0.0,0,0,0.0


ForkEvent
github_lakehouse.silver.fork_events


event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
ForkEvent,event_id,github_lakehouse.silver.fork_events,1,id,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_type,github_lakehouse.silver.fork_events,2,type,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_timestamp,github_lakehouse.silver.fork_events,3,created_at,TIMESTAMP,"G5, G6",0,0.0,0,0,0.0
ForkEvent,event_date,github_lakehouse.silver.fork_events,4,derived: DATE(event_timestamp),DATE,"G5, G6",0,0.0,0,0,0.0
ForkEvent,actor_id,github_lakehouse.silver.fork_events,5,actor.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ForkEvent,actor_login,github_lakehouse.silver.fork_events,6,actor.login,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,repo_id,github_lakehouse.silver.fork_events,7,repo.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ForkEvent,repo_name,github_lakehouse.silver.fork_events,8,repo.name,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,_source_file,github_lakehouse.silver.fork_events,9,Bronze column,STRING,"G5, G6",0,0.0,0,0,0.0
ForkEvent,_source_date,github_lakehouse.silver.fork_events,10,Bronze column,DATE,"G5, G6",0,0.0,0,0,0.0


ReleaseEvent
github_lakehouse.silver.release_events


event_type,silver_field,silver_table,field_order,source_path,silver_type,gold_dependency,null_count,null_pct,blank_string_count,cast_failures,cast_failure_pct
ReleaseEvent,event_id,github_lakehouse.silver.release_events,1,id,STRING,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,event_type,github_lakehouse.silver.release_events,2,type,STRING,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,event_timestamp,github_lakehouse.silver.release_events,3,created_at,TIMESTAMP,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,event_date,github_lakehouse.silver.release_events,4,derived: DATE(event_timestamp),DATE,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,actor_id,github_lakehouse.silver.release_events,5,actor.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,actor_login,github_lakehouse.silver.release_events,6,actor.login,STRING,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,repo_id,github_lakehouse.silver.release_events,7,repo.id,BIGINT,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,repo_name,github_lakehouse.silver.release_events,8,repo.name,STRING,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,_source_file,github_lakehouse.silver.release_events,9,Bronze column,STRING,"G5, G6",0,0.0,0,0,0.0
ReleaseEvent,_source_date,github_lakehouse.silver.release_events,10,Bronze column,DATE,"G5, G6",0,0.0,0,0,0.0


## 13. Quick Problem Summary

The previous sections contain the complete profiles.

The following cells show only records or fields that deserve investigation.

In [0]:
display(
    cast_profile_df
    .filter(
        F.col("cast_failures") > 0
    )
    .orderBy(
        F.desc("cast_failures")
    )
)

event_type,field,target_type,source_non_null,cast_failures,cast_failure_pct


In [0]:
display(
    null_profile_df
    .filter(
        (F.col("null_count") > 0)
        | (F.col("blank_string_count") > 0)
    )
    .orderBy(
        F.desc("null_pct")
    )
)

event_type,field,silver_type,total_rows,null_count,null_pct,blank_string_count
PullRequestEvent,pr_merged_at,TIMESTAMP,2173964,1369201,62.9818,0
IssuesEvent,issue_closed_at,TIMESTAMP,548959,330119,60.1355,0
PullRequestEvent,pr_closed_at,TIMESTAMP,2173964,1136804,52.2918,0
ReleaseEvent,release_published_at,TIMESTAMP,159416,278,0.1744,0
IssueCommentEvent,comment_action,STRING,1357327,2,1.0E-4,0
IssueCommentEvent,issue_id,BIGINT,1357327,2,1.0E-4,0
IssueCommentEvent,issue_number,BIGINT,1357327,2,1.0E-4,0
IssueCommentEvent,comment_target_type,STRING,1357327,2,1.0E-4,0
IssueCommentEvent,comment_id,BIGINT,1357327,2,1.0E-4,0
ReleaseEvent,release_tag,STRING,159416,0,0.0,1


In [0]:
display(
    logical_check_df
    .filter(
        F.col("problem_count") > 0
    )
    .orderBy(
        F.desc("problem_count")
    )
)

event_type,check_name,problem_count
PushEvent,distinct_commits_greater_than_total_commits,611
ReleaseEvent,published_release_missing_published_at,278


In [0]:
display(
    duplicate_summary
    .filter(
        (F.col("duplicate_event_ids") > 0)
        | (F.col("null_event_ids") > 0)
    )
)

event_type,total_rows,distinct_event_ids,null_event_ids,duplicate_event_ids,extra_duplicate_rows
ForkEvent,326419,326417,0,2,2
IssueCommentEvent,1357327,1357324,0,3,3
IssuesEvent,548959,548956,0,2,3
PullRequestEvent,2173964,2173958,0,5,6
PullRequestReviewEvent,717285,717277,0,4,8
PushEvent,26537024,26536967,0,56,57
WatchEvent,1380565,1380559,0,6,6


# Phase 1 Complete

After reviewing the profiling results, we should be able to answer:

- Which fields are always present?
- Which NULLs are legitimate?
- Which fields should be required?
- Which fields are conditionally required?
- Are any datatype conversions unsafe?
- Do duplicate event IDs exist?
- Are duplicates identical or conflicting?
- Which action values actually exist?
- Which review states actually exist?
- Are there malformed JSON records?
- Are there logical inconsistencies?
- Which releases have complete ±24-hour observation windows?

Once those decisions are locked, Phase 2 will build the Lakeflow Declarative Pipeline
that produces the 9 clean Silver tables.